<a href="https://colab.research.google.com/github/Priya-Kumari-Chourasia/deep_learning/blob/main/bahdanau_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)

PyTorch: 2.11.0+cpu
Device: cpu


In [2]:
data = [
    ("how are you", "आप कैसे हैं"),
    ("i am fine", "मैं ठीक हूँ"),
    ("what is your name", "आपका नाम क्या है"),
    ("my name is ram", "मेरा नाम राम है"),
    ("where are you going", "आप कहाँ जा रहे हैं"),
    ("i love india", "मुझे भारत से प्यार है"),
    ("i am happy", "मैं खुश हूँ"),
    ("i am sad", "मैं दुखी हूँ"),
    ("he is happy", "वह खुश है"),
    ("she is happy", "वह खुश है"),
    ("this is my house", "यह मेरा घर है"),
    ("this is my book", "यह मेरी किताब है"),
    ("i like tea", "मुझे चाय पसंद है"),
    ("i like coffee", "मुझे कॉफी पसंद है"),
    ("good morning", "सुप्रभात"),
    ("good night", "शुभ रात्रि"),
    ("thank you", "धन्यवाद"),
    ("please help me", "कृपया मेरी मदद करें"),
    ("what are you doing", "आप क्या कर रहे हैं"),
    ("i am learning", "मैं सीख रहा हूँ")
]

print("Number of sentence pairs:", len(data))

Number of sentence pairs: 20


In [3]:
english_sentences = [x[0] for x in data]
hindi_sentences = [x[1] for x in data]

print("English:", english_sentences[0])
print("Hindi  :", hindi_sentences[0])

English: how are you
Hindi  : आप कैसे हैं


In [4]:
def build_vocab(sentences):

    vocab = {
        "<pad>": 0,
        "<sos>": 1,
        "<eos>": 2,
        "<unk>": 3
    }

    for sentence in sentences:

        words = sentence.lower().split()

        for word in words:

            if word not in vocab:
                vocab[word] = len(vocab)

    return vocab

In [5]:
src_vocab = build_vocab(english_sentences)
trg_vocab = build_vocab(hindi_sentences)

print("English vocabulary size:", len(src_vocab))
print("Hindi vocabulary size:", len(trg_vocab))

English vocabulary size: 39
Hindi vocabulary size: 43


In [6]:
print("ENGLISH VOCABULARY")

for word, number in src_vocab.items():
    print(number, "->", word)

ENGLISH VOCABULARY
0 -> <pad>
1 -> <sos>
2 -> <eos>
3 -> <unk>
4 -> how
5 -> are
6 -> you
7 -> i
8 -> am
9 -> fine
10 -> what
11 -> is
12 -> your
13 -> name
14 -> my
15 -> ram
16 -> where
17 -> going
18 -> love
19 -> india
20 -> happy
21 -> sad
22 -> he
23 -> she
24 -> this
25 -> house
26 -> book
27 -> like
28 -> tea
29 -> coffee
30 -> good
31 -> morning
32 -> night
33 -> thank
34 -> please
35 -> help
36 -> me
37 -> doing
38 -> learning


In [7]:
src_itos = {number: word for word, number in src_vocab.items()}
trg_itos = {number: word for word, number in trg_vocab.items()}

In [8]:
def numericalize(sentence, vocab):

    words = sentence.lower().split()

    ids = [vocab["<sos>"]]

    for word in words:

        if word in vocab:
            ids.append(vocab[word])
        else:
            ids.append(vocab["<unk>"])

    ids.append(vocab["<eos>"])

    return ids

In [9]:
def create_tensor(sentences, vocab):

    tensor_list = []

    for sentence in sentences:

        ids = numericalize(sentence, vocab)

        tensor = torch.tensor(
            ids,
            dtype=torch.long
        )

        tensor_list.append(tensor)

    return tensor_list

In [10]:
src_data = create_tensor(
    english_sentences,
    src_vocab
)

trg_data = create_tensor(
    hindi_sentences,
    trg_vocab
)

In [11]:
src_tensor = pad_sequence(
    src_data,
    padding_value=src_vocab["<pad>"]
)

trg_tensor = pad_sequence(
    trg_data,
    padding_value=trg_vocab["<pad>"]
)

In [12]:
print("Source shape:", src_tensor.shape)
print("Target shape:", trg_tensor.shape)

Source shape: torch.Size([6, 20])
Target shape: torch.Size([7, 20])


In [13]:
src = src_tensor.to(device)
trg = trg_tensor.to(device)

print("SRC:", src.shape)
print("TRG:", trg.shape)

SRC: torch.Size([6, 20])
TRG: torch.Size([7, 20])


In [14]:
class Encoder(nn.Module):

    def __init__(
        self,
        input_dim,
        embedding_dim,
        hidden_dim
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            input_dim,
            embedding_dim
        )

        self.rnn = nn.LSTM(
            embedding_dim,
            hidden_dim
        )

    def forward(self, src):

        embedded = self.embedding(src)

        outputs, (hidden, cell) = self.rnn(
            embedded
        )

        return outputs, hidden, cell

In [15]:
class BahdanauAttention(nn.Module):

    def __init__(
        self,
        encoder_hidden_dim,
        decoder_hidden_dim,
        attention_dim
    ):

        super().__init__()

        self.attention = nn.Linear(
            encoder_hidden_dim + decoder_hidden_dim,
            attention_dim
        )

        self.v = nn.Linear(
            attention_dim,
            1,
            bias=False
        )

    def forward(
        self,
        decoder_hidden,
        encoder_outputs
    ):

        # encoder_outputs:
        # [src_len, batch_size, encoder_hidden_dim]

        src_len = encoder_outputs.shape[0]

        # decoder_hidden:
        # [batch_size, decoder_hidden_dim]

        decoder_hidden = decoder_hidden.unsqueeze(1)

        # [batch_size, 1, decoder_hidden_dim]

        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        # [batch_size, src_len, encoder_hidden_dim]

        decoder_hidden = decoder_hidden.repeat(
            1,
            src_len,
            1
        )

        # [batch_size, src_len, decoder_hidden_dim]

        energy_input = torch.cat(
            (
                encoder_outputs,
                decoder_hidden
            ),
            dim=2
        )

        # [batch_size, src_len,
        #  encoder_hidden_dim + decoder_hidden_dim]

        energy = torch.tanh(
            self.attention(energy_input)
        )

        # [batch_size, src_len, attention_dim]

        attention = self.v(energy).squeeze(2)

        # [batch_size, src_len]

        return torch.softmax(
            attention,
            dim=1
        )

In [18]:
class Decoder(nn.Module):

    def __init__(
        self,
        output_dim,
        embedding_dim,
        encoder_hidden_dim,
        decoder_hidden_dim,
        attention
    ):
        super().__init__()

        self.output_dim = output_dim
        self.attention = attention

        self.embedding = nn.Embedding(
            output_dim,
            embedding_dim
        )

        self.rnn = nn.LSTM(
            embedding_dim + encoder_hidden_dim,
            decoder_hidden_dim
        )

        self.fc = nn.Linear(
            decoder_hidden_dim + encoder_hidden_dim + embedding_dim,
            output_dim
        )

    def forward(
        self,
        input_token,
        hidden,
        cell,
        encoder_outputs
    ):

        input_token = input_token.unsqueeze(0)

        embedded = self.embedding(input_token)

        embedded = embedded.permute(1, 0, 2)

        attention_weights = self.attention(
            hidden[-1],
            encoder_outputs
        )

        attention_weights = attention_weights.unsqueeze(1)

        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        context = torch.bmm(
            attention_weights,
            encoder_outputs
        )

        rnn_input = torch.cat(
            (
                embedded,
                context
            ),
            dim=2
        )

        rnn_input = rnn_input.permute(1, 0, 2)

        output, (hidden, cell) = self.rnn(
            rnn_input,
            (hidden, cell)
        )

        output = output.squeeze(0)
        context = context.squeeze(1)
        embedded = embedded.squeeze(1)

        prediction = self.fc(
            torch.cat(
                (
                    output,
                    context,
                    embedded
                ),
                dim=1
            )
        )

        return (
            prediction,
            hidden,
            cell,
            attention_weights.squeeze(1)
        )

In [23]:
class Seq2Seq(nn.Module):

    def __init__(
        self,
        encoder,
        decoder,
        device
    ):

        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(
        self,
        src,
        trg,
        teacher_forcing_ratio=0.5
    ):

        batch_size = src.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(
            trg_len,
            batch_size,
            trg_vocab_size
        ).to(self.device)

        encoder_outputs, hidden, cell = self.encoder(src)

        input_token = trg[0, :]

        for t in range(1, trg_len):

            output, hidden, cell, attention_weights = self.decoder(
                input_token,
                hidden,
                cell,
                encoder_outputs
            )

            outputs[t] = output

            best_prediction = output.argmax(1)

            if random.random() < teacher_forcing_ratio:
                input_token = trg[t, :]
            else:
                input_token = best_prediction

        return outputs

In [24]:
INPUT_DIM = len(src_vocab)
OUTPUT_DIM = len(trg_vocab)

ENC_EMB_DIM = 64
DEC_EMB_DIM = 64

ENC_HIDDEN_DIM = 128
DEC_HIDDEN_DIM = 128

ATTENTION_DIM = 64

In [25]:
encoder = Encoder(
    INPUT_DIM,
    ENC_EMB_DIM,
    ENC_HIDDEN_DIM
).to(device)

attention = BahdanauAttention(
    ENC_HIDDEN_DIM,
    DEC_HIDDEN_DIM,
    ATTENTION_DIM
).to(device)

decoder = Decoder(
    OUTPUT_DIM,
    DEC_EMB_DIM,
    ENC_HIDDEN_DIM,
    DEC_HIDDEN_DIM,
    attention
).to(device)

model = Seq2Seq(
    encoder,
    decoder,
    device
).to(device)

In [28]:
import random
output = model(src, trg)

print("SRC:", src.shape)
print("TRG:", trg.shape)
print("OUTPUT:", output.shape)

SRC: torch.Size([6, 20])
TRG: torch.Size([7, 20])
OUTPUT: torch.Size([7, 20, 43])


In [30]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss(
    ignore_index=trg_vocab["<pad>"]
)

In [31]:
def train(
    model,
    src,
    trg,
    optimizer,
    criterion,
    epochs
):

    model.train()

    for epoch in range(epochs):

        optimizer.zero_grad()

        output = model(
            src,
            trg,
            teacher_forcing_ratio=0.5
        )

        output_dim = output.shape[-1]

        output = output[1:].reshape(
            -1,
            output_dim
        )

        trg_y = trg[1:].reshape(-1)

        loss = criterion(
            output,
            trg_y
        )

        loss.backward()

        optimizer.step()

        if (epoch + 1) % 50 == 0:

            print(
                f"Epoch: {epoch + 1:3d} | "
                f"Loss: {loss.item():.4f}"
            )

In [32]:
train(
    model,
    src,
    trg,
    optimizer,
    criterion,
    epochs=1000
)

Epoch:  50 | Loss: 0.3750
Epoch: 100 | Loss: 0.0337
Epoch: 150 | Loss: 0.0146
Epoch: 200 | Loss: 0.0086
Epoch: 250 | Loss: 0.0058
Epoch: 300 | Loss: 0.0042
Epoch: 350 | Loss: 0.0032
Epoch: 400 | Loss: 0.0026
Epoch: 450 | Loss: 0.0021
Epoch: 500 | Loss: 0.0017
Epoch: 550 | Loss: 0.0015
Epoch: 600 | Loss: 0.0013
Epoch: 650 | Loss: 0.0011
Epoch: 700 | Loss: 0.0010
Epoch: 750 | Loss: 0.0009
Epoch: 800 | Loss: 0.0008
Epoch: 850 | Loss: 0.0007
Epoch: 900 | Loss: 0.0006
Epoch: 950 | Loss: 0.0006
Epoch: 1000 | Loss: 0.0005


In [33]:
def translate_sentence(
    model,
    sentence,
    src_vocab,
    trg_itos,
    max_length=20
):

    model.eval()

    # Convert English sentence to token IDs
    tokens = sentence.lower().split()

    src_ids = [src_vocab["<sos>"]]

    for token in tokens:
        if token in src_vocab:
            src_ids.append(src_vocab[token])
        else:
            src_ids.append(src_vocab["<unk>"])

    src_ids.append(src_vocab["<eos>"])

    # Convert to tensor
    src_tensor = torch.tensor(
        src_ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(1)

    # Encoder
    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(
            src_tensor
        )

    # Start decoder with <sos>
    input_token = torch.tensor(
        [trg_vocab["<sos>"]],
        dtype=torch.long,
        device=device
    )

    translated_words = []

    for _ in range(max_length):

        with torch.no_grad():

            output, hidden, cell, attention = model.decoder(
                input_token,
                hidden,
                cell,
                encoder_outputs
            )

        # Select most probable word
        predicted_token = output.argmax(1).item()

        # Stop if <eos>
        if predicted_token == trg_vocab["<eos>"]:
            break

        # Convert ID → Hindi word
        predicted_word = trg_itos[predicted_token]

        translated_words.append(predicted_word)

        # Feed predicted word back into decoder
        input_token = torch.tensor(
            [predicted_token],
            dtype=torch.long,
            device=device
        )

    return " ".join(translated_words)

In [34]:
translation = translate_sentence(
    model,
    "how are you",
    src_vocab,
    trg_itos
)

print("Translation:", translation)

Translation: आप कैसे हैं
